In [ ]:
import os
import glob
import torch
import numpy as np
import pandas as pd

import physicsnemo
from physicsnemo.models.mlp.fully_connected import FullyConnected

BASE_PATH = os.getcwd()

DATASET_DIR = os.path.join(BASE_PATH, "dataset")
MODEL_DIR = os.path.join(BASE_PATH, "models")
FIG_DIR = os.path.join(BASE_PATH, "figures")

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Target Working Directory: {BASE_PATH}")
print(f"Executing training on device: {device}")

csv_files = glob.glob(os.path.join(DATASET_DIR, "*.csv")) + glob.glob(os.path.join(BASE_PATH, "*.csv")) + glob.glob(os.path.join(BASE_PATH, "**", "*.csv"), recursive=True)

csv_files = list(dict.fromkeys(csv_files))

if not csv_files:
    raise FileNotFoundError(
        f"No CSV dataset found inside '{BASE_PATH}'. "
        "Verify that 'ald_spatial_dataset_n2500.csv' is synced into the 'dataset' subfolder."
    )

DATASET_CSV = sorted(csv_files, key=os.path.getmtime, reverse=True)[0]
print(f"Loaded dataset file: {DATASET_CSV}")

df = pd.read_csv(DATASET_CSV)
N_TOTAL_SAMPLES = len(df)
print(f"Total dataset records: {N_TOTAL_SAMPLES}")

FEATURE_COLS = ["temperature_C", "pulse_time_s", "num_cycles", "tma_mole_fraction"]
TARGET_COLS = [col for col in df.columns if col.startswith("thickness_pt_")]
N_SPATIAL_POINTS = len(TARGET_COLS)

if N_SPATIAL_POINTS == 0:
    raise ValueError("No spatial thickness columns ('thickness_pt_*') detected in the dataset!")

print(f"Detected {N_SPATIAL_POINTS} spatial target coordinates per sample.")

X = df[FEATURE_COLS].values.astype(np.float32)
Y = df[TARGET_COLS].values.astype(np.float32)

X_mean, X_std = X.mean(0), X.std(0) + 1e-8
Y_mean, Y_std = Y.mean(0), Y.std(0) + 1e-8
X_norm = (X - X_mean) / X_std
Y_norm = (Y - Y_mean) / Y_std

X_mean_t = torch.tensor(X_mean, device=device)
X_std_t = torch.tensor(X_std, device=device)
Y_mean_t = torch.tensor(Y_mean, device=device)
Y_std_t = torch.tensor(Y_std, device=device)

n_train = int(0.8 * len(X_norm))
perm = np.random.permutation(len(X_norm))
train_idx, val_idx = perm[:n_train], perm[n_train:]

X_train = torch.tensor(X_norm[train_idx], device=device, requires_grad=True)
Y_train = torch.tensor(Y_norm[train_idx], device=device)
X_val = torch.tensor(X_norm[val_idx], device=device)
Y_val = torch.tensor(Y_norm[val_idx], device=device)

GPC_train = torch.tensor(df["gpc_nm_per_cycle"].values[train_idx].astype(np.float32), device=device)

surrogate = FullyConnected(
    in_features=len(FEATURE_COLS),
    out_features=N_SPATIAL_POINTS,
    num_layers=6,
    layer_size=128,
).to(device)

optimizer = torch.optim.Adam(surrogate.parameters(), lr=2e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=3000)
loss_fn = torch.nn.MSELoss()

model_ckpt_path = os.path.join(MODEL_DIR, "ald_spatial_pinn_checkpoint.pt")

start_epoch = 0
train_losses, val_losses = [], []

if os.path.exists(model_ckpt_path):
    print(f"Found active checkpoint. Resuming PINN training from {model_ckpt_path}...")
    checkpoint = torch.load(model_ckpt_path, map_location=device)
    surrogate.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    train_losses = checkpoint.get("train_losses", [])
    val_losses = checkpoint.get("val_losses", [])
    X_mean, X_std = checkpoint["X_mean"], checkpoint["X_std"]
    Y_mean, Y_std = checkpoint["Y_mean"], checkpoint["Y_std"]
else:
    print("No checkpoint found. Initializing PINN training from scratch.")

N_EPOCHS = 3000
LAMBDA_PHYSICS = 0.5

boundary_width = max(1, int(0.05 * N_SPATIAL_POINTS))
top_indices = list(range(0, boundary_width)) + list(range(N_SPATIAL_POINTS - boundary_width, N_SPATIAL_POINTS))

print(f"Enforcing Cantera GPC derivative constraint on top fin indices: {top_indices}")

for epoch in range(start_epoch, N_EPOCHS):
    surrogate.train()
    optimizer.zero_grad()

    pred_norm = surrogate(X_train)
    loss_data = loss_fn(pred_norm, Y_train)

    pred = pred_norm * Y_std_t + Y_mean_t

    T_top = pred[:, top_indices].mean(dim=1)

    dTtop_dX = torch.autograd.grad(
        outputs=T_top, inputs=X_train,
        grad_outputs=torch.ones_like(T_top),
        create_graph=True, retain_graph=True
    )[0]

    dTtop_dN = dTtop_dX[:, 2] / X_std_t[2]
    loss_kinetics = torch.mean((dTtop_dN - GPC_train) ** 2)

    T_mean_global = pred.mean(dim=1)
    dTglobal_dX = torch.autograd.grad(
        outputs=T_mean_global, inputs=X_train,
        grad_outputs=torch.ones_like(T_mean_global),
        create_graph=True, retain_graph=True
    )[0]

    dTglobal_dN = dTglobal_dX[:, 2] / X_std_t[2]
    d2Tglobal_dX2 = torch.autograd.grad(
        outputs=dTglobal_dN, inputs=X_train,
        grad_outputs=torch.ones_like(dTglobal_dN),
        create_graph=True, retain_graph=True
    )[0]

    d2Tglobal_dN2 = d2Tglobal_dX2[:, 2] / X_std_t[2]
    loss_linear = torch.mean(d2Tglobal_dN2 ** 2)

    X_zero = X_train.clone().detach()
    X_zero[:, 2] = float((0.0 - X_mean[2]) / X_std[2])
    pred_zero = surrogate(X_zero)
    T_zero = pred_zero * Y_std_t + Y_mean_t
    loss_origin = torch.mean(T_zero ** 2)

    loss_physics = loss_kinetics + loss_linear + loss_origin

    loss = loss_data + (LAMBDA_PHYSICS * loss_physics)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(surrogate.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()

    surrogate.eval()
    with torch.no_grad():
        val_pred = surrogate(X_val)
        val_loss = loss_fn(val_pred, Y_val)

    train_losses.append(loss.item())
    val_losses.append(val_loss.item())

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:4d}/{N_EPOCHS} | Total Loss: {loss.item():.5f} "
              f"(Data: {loss_data.item():.5f}, Phys: {loss_physics.item():.5f}) | Val Loss: {val_loss.item():.5f}")

        torch.save({
            "epoch": epoch,
            "model_state_dict": surrogate.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "train_losses": train_losses,
            "val_losses": val_losses,
            "X_mean": X_mean, "X_std": X_std,
            "Y_mean": Y_mean, "Y_std": Y_std,
        }, model_ckpt_path)

print("\nPINN Training Loop Complete.")


In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

print("=" * 70)
print("EXTRACTING ALL TRAINING ARTIFACTS, WEIGHTS, AND DATASET METRICS")
print("=" * 70)

weights_file = os.path.join(MODEL_DIR, "ald_pinn_final_weights.pt")
bundle_file = os.path.join(MODEL_DIR, "ald_pinn_deployment_bundle.pt")

torch.save(surrogate.state_dict(), weights_file)

deployment_bundle = {
    "model_state_dict": surrogate.state_dict(),
    "architecture": {
        "in_features": len(FEATURE_COLS),
        "out_features": N_SPATIAL_POINTS,
        "num_layers": 6,
        "layer_size": 128
    },
    "preprocessing": {
        "feature_cols": FEATURE_COLS,
        "target_cols": TARGET_COLS,
        "X_mean": X_mean,
        "X_std": X_std,
        "Y_mean": Y_mean,
        "Y_std": Y_std
    },
    "training_meta": {
        "total_samples": N_TOTAL_SAMPLES,
        "train_samples": len(train_idx),
        "val_samples": len(val_idx),
        "epochs_completed": len(train_losses)
    }
}
torch.save(deployment_bundle, bundle_file)
print(f"[SAVED] Raw weights saved to: {weights_file}")
print(f"[SAVED] Full deployment bundle saved to: {bundle_file}")

print("\nCalculating dataset statistics and correlation matrices...")

eda_summary = df.describe().T
eda_summary["skewness"] = df.skew(numeric_only=True)
eda_summary["kurtosis"] = df.kurtosis(numeric_only=True)

eda_csv_path = os.path.join(DATASET_DIR, "dataset_statistical_summary.csv")
eda_summary.to_csv(eda_csv_path)
print(f"[SAVED] Complete dataset statistical summary saved to: {eda_csv_path}")

corr_matrix = df.corr(numeric_only=True)
corr_csv_path = os.path.join(DATASET_DIR, "dataset_correlation_matrix.csv")
corr_matrix.to_csv(corr_csv_path)
print(f"[SAVED] Feature correlation matrix saved to: {corr_csv_path}")

if len(train_losses) > 0:
    history_df = pd.DataFrame({
        "epoch": range(1, len(train_losses) + 1),
        "train_loss_total": train_losses,
        "val_loss": val_losses
    })
    history_csv_path = os.path.join(DATASET_DIR, "training_history_log.csv")
    history_df.to_csv(history_csv_path, index=False)
    print(f"[SAVED] Epoch-by-epoch training history saved to: {history_csv_path}")

surrogate.eval()
with torch.no_grad():
    y_pred_val_norm = surrogate(X_val).cpu().numpy()
    y_true_val_norm = Y_val.cpu().numpy()

y_pred_val = y_pred_val_norm * Y_std + Y_mean
y_true_val = y_true_val_norm * Y_std + Y_mean

abs_spatial_errors = np.abs(y_true_val - y_pred_val)

pointwise_metrics = []
for j in range(N_SPATIAL_POINTS):
    true_pt = y_true_val[:, j]
    pred_pt = y_pred_val[:, j]

    mae_j = mean_absolute_error(true_pt, pred_pt)
    rmse_j = np.sqrt(mean_squared_error(true_pt, pred_pt))
    r2_j = r2_score(true_pt, pred_pt)
    max_err_j = np.max(np.abs(true_pt - pred_pt))
    mape_j = np.mean(np.abs((true_pt - pred_pt) / np.clip(true_pt, 1e-6, None))) * 100.0

    pointwise_metrics.append({
        "spatial_point_index": j,
        "coordinate_name": TARGET_COLS[j],
        "mae_nm": float(mae_j),
        "rmse_nm": float(rmse_j),
        "r2_score": float(r2_j),
        "max_error_nm": float(max_err_j),
        "mape_percent": float(mape_j)
    })

pointwise_df = pd.DataFrame(pointwise_metrics)
pointwise_csv_path = os.path.join(DATASET_DIR, "spatial_pointwise_metrics.csv")
pointwise_df.to_csv(pointwise_csv_path, index=False)
print(f"[SAVED] Per-point spatial accuracy metrics saved to: {pointwise_csv_path}")

boundary_w = max(1, int(0.05 * N_SPATIAL_POINTS))
top_pt_indices = list(range(0, boundary_w)) + list(range(N_SPATIAL_POINTS - boundary_w, N_SPATIAL_POINTS))

trench_center_start = int(0.45 * N_SPATIAL_POINTS)
trench_center_end = int(0.55 * N_SPATIAL_POINTS)
bottom_pt_indices = list(range(trench_center_start, trench_center_end))

true_top_thick = y_true_val[:, top_pt_indices].mean(axis=1)
pred_top_thick = y_pred_val[:, top_pt_indices].mean(axis=1)

true_bottom_thick = y_true_val[:, bottom_pt_indices].mean(axis=1)
pred_bottom_thick = y_pred_val[:, bottom_pt_indices].mean(axis=1)

true_conformality = true_bottom_thick / np.clip(true_top_thick, 1e-6, None)
pred_conformality = pred_bottom_thick / np.clip(pred_top_thick, 1e-6, None)

val_sample_records = []
val_df_orig = df.iloc[val_idx].reset_index(drop=True)

for i in range(len(val_idx)):
    val_sample_records.append({
        "original_sample_index": int(val_idx[i]),
        "temperature_C": float(val_df_orig.loc[i, "temperature_C"]),
        "pulse_time_s": float(val_df_orig.loc[i, "pulse_time_s"]),
        "num_cycles": int(val_df_orig.loc[i, "num_cycles"]),
        "tma_mole_fraction": float(val_df_orig.loc[i, "tma_mole_fraction"]),
        "true_top_thickness_nm": float(true_top_thick[i]),
        "true_bottom_thickness_nm": float(true_bottom_thick[i]),
        "true_conformality_ratio": float(true_conformality[i]),
        "pred_top_thickness_nm": float(pred_top_thick[i]),
        "pred_bottom_thickness_nm": float(pred_bottom_thick[i]),
        "pred_conformality_ratio": float(pred_conformality[i]),
        "top_thickness_error_nm": float(abs(true_top_thick[i] - pred_top_thick[i])),
        "bottom_thickness_error_nm": float(abs(true_bottom_thick[i] - pred_bottom_thick[i])),
        "conformality_abs_error": float(abs(true_conformality[i] - pred_conformality[i])),
        "sample_mean_spatial_mae_nm": float(mean_absolute_error(y_true_val[i], y_pred_val[i]))
    })

per_sample_df = pd.DataFrame(val_sample_records)
per_sample_csv_path = os.path.join(DATASET_DIR, "per_sample_validation_eval.csv")
per_sample_df.to_csv(per_sample_csv_path, index=False)
print(f"[SAVED] Per-sample physical evaluation data saved to: {per_sample_csv_path}")

global_r2_val = float(r2_score(y_true_val.flatten(), y_pred_val.flatten()))
global_mae_val = float(mean_absolute_error(y_true_val, y_pred_val))
global_rmse_val = float(np.sqrt(mean_squared_error(y_true_val, y_pred_val)))
global_max_err = float(np.max(abs_spatial_errors))
global_mape_val = float(np.mean(abs_spatial_errors / np.clip(y_true_val, 1e-6, None)) * 100.0)

conformality_r2 = float(r2_score(true_conformality, pred_conformality))
conformality_mae = float(mean_absolute_error(true_conformality, pred_conformality))

summary_report = {
    "model_name": "PhysicsNeMo_ALD_Spatial_PINN",
    "dataset_info": {
        "total_records": N_TOTAL_SAMPLES,
        "train_records": len(train_idx),
        "validation_records": len(val_idx),
        "spatial_points_per_sample": N_SPATIAL_POINTS
    },
    "metrics_global": {
        "spatial_profile_r2": global_r2_val,
        "spatial_profile_mae_nm": global_mae_val,
        "spatial_profile_rmse_nm": global_rmse_val,
        "spatial_profile_max_error_nm": global_max_err,
        "spatial_profile_mape_percent": global_mape_val
    },
    "metrics_physical_conformality": {
        "conformality_r2_score": conformality_r2,
        "conformality_mae": conformality_mae,
        "top_thickness_mae_nm": float(mean_absolute_error(true_top_thick, pred_top_thick)),
        "bottom_thickness_mae_nm": float(mean_absolute_error(true_bottom_thick, pred_bottom_thick))
    },
    "training_convergence": {
        "final_train_loss": float(train_losses[-1]) if train_losses else None,
        "final_val_loss": float(val_losses[-1]) if val_losses else None,
        "total_epochs_trained": len(train_losses)
    }
}

summary_json_path = os.path.join(BASE_PATH, "pinn_summary_report.json")
with open(summary_json_path, "w") as f:
    json.dump(summary_report, f, indent=4)
print(f"[SAVED] Comprehensive JSON summary report saved to: {summary_json_path}")

def plot_spatial_heatmaps(y_true, y_pred, save_path, r2_val):
    abs_error = np.abs(y_true - y_pred)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), gridspec_kw={'wspace': 0.15})

    vmin = min(y_true.min(), y_pred.min())
    vmax = max(y_true.max(), y_pred.max())

    im0 = axes[0].imshow(y_true, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
    axes[0].set_title("Ground Truth Thickness (nm)", fontweight='bold')
    axes[0].set_xlabel(f"Spatial Coordinate Index (0 -> {N_SPATIAL_POINTS-1})")
    axes[0].set_ylabel("Validation Sample Index")
    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(y_pred, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
    axes[1].set_title("PINN Predicted Thickness (nm)", fontweight='bold')
    axes[1].set_xlabel(f"Spatial Coordinate Index (0 -> {N_SPATIAL_POINTS-1})")
    axes[1].set_yticks([])
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    im2 = axes[2].imshow(abs_error, aspect='auto', cmap='inferno')
    axes[2].set_title("Absolute Error (nm)", fontweight='bold')
    axes[2].set_xlabel(f"Spatial Coordinate Index (0 -> {N_SPATIAL_POINTS-1})")
    axes[2].set_yticks([])
    fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    plt.suptitle(f"Spatial Profile Verification (N={N_TOTAL_SAMPLES}, Global R² = {r2_val:.4f})", fontsize=16)
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.close()

heatmap_path = os.path.join(FIG_DIR, f"spatial_heatmaps_n{N_TOTAL_SAMPLES}.png")
plot_spatial_heatmaps(y_true_val, y_pred_val, heatmap_path, global_r2_val)
print(f"[SAVED] Spatial heatmaps saved to: {heatmap_path}")

fig, axes = plt.subplots(2, 2, figsize=(16, 12), gridspec_kw={'hspace': 0.25, 'wspace': 0.2})

axes[0, 0].plot(train_losses, label="Total Train Loss (Data + Physics)", color="#1f77b4", linewidth=1.5)
axes[0, 0].plot(val_losses, label="Validation Loss", color="#ff7f0e", linewidth=1.5, linestyle="--")
axes[0, 0].set_yscale("log")
axes[0, 0].set_xlabel("Epoch", fontweight='bold')
axes[0, 0].set_ylabel("Loss (MSE Log Scale)", fontweight='bold')
axes[0, 0].set_title("PINN Loss Convergence", fontweight='bold', fontsize=12)
axes[0, 0].grid(True, which="both", alpha=0.3)
axes[0, 0].legend()

spatial_indices = np.arange(N_SPATIAL_POINTS)
mae_by_point = pointwise_df["mae_nm"].values
std_by_point = abs_spatial_errors.std(axis=0)

axes[0, 1].plot(spatial_indices, mae_by_point, color="#d62728", linewidth=2.0, label="Mean Absolute Error")
axes[0, 1].fill_between(spatial_indices,
                        np.maximum(0, mae_by_point - std_by_point),
                        mae_by_point + std_by_point,
                        color="#d62728", alpha=0.2, label="±1 Std Dev Uncertainty")
axes[0, 1].set_xlabel("Spatial Point Index (0 = Fin Peak, 50 = Trench Bottom)", fontweight='bold')
axes[0, 1].set_ylabel("Absolute Error (nm)", fontweight='bold')
axes[0, 1].set_title("Pointwise Spatial Error Profile Across Geometry", fontweight='bold', fontsize=12)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

axes[1, 0].scatter(y_true_val.flatten(), y_pred_val.flatten(), alpha=0.3, color="#2ca02c", s=10)
min_val = min(y_true_val.min(), y_pred_val.min())
max_val = max(y_true_val.max(), y_pred_val.max())
axes[1, 0].plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.7, label="1:1 Perfect Parity")
axes[1, 0].set_xlabel("Ground Truth Thickness (nm)", fontweight='bold')
axes[1, 0].set_ylabel("PINN Predicted Thickness (nm)", fontweight='bold')
axes[1, 0].set_title(f"Global Spatial Thickness Parity (R² = {global_r2_val:.4f})", fontweight='bold', fontsize=12)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

axes[1, 1].scatter(true_conformality, pred_conformality, alpha=0.6, color="#9467bd", s=25)
c_min = min(true_conformality.min(), pred_conformality.min())
c_max = max(true_conformality.max(), pred_conformality.max())
axes[1, 1].plot([c_min, c_max], [c_min, c_max], 'k--', alpha=0.7, label="1:1 Perfect Parity")
axes[1, 1].set_xlabel("Ground Truth Conformality (Bottom / Top)", fontweight='bold')
axes[1, 1].set_ylabel("PINN Predicted Conformality", fontweight='bold')
axes[1, 1].set_title(f"Conformality Ratio Parity (R² = {conformality_r2:.4f}, MAE = {conformality_mae:.4f})", fontweight='bold', fontsize=12)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.suptitle(f"PINN Model Comprehensive Diagnostics (Dataset N = {N_TOTAL_SAMPLES})", fontsize=16, fontweight='bold', y=0.98)
master_fig_path = os.path.join(FIG_DIR, f"pinn_comprehensive_diagnostics_n{N_TOTAL_SAMPLES}.png")
plt.savefig(master_fig_path, dpi=200, bbox_inches='tight')
plt.close()

print(f"[SAVED] Master diagnostic figure saved to: {master_fig_path}")

print("\n" + "=" * 70)
print("POST-TRAINING ARTIFACT SERIALIZATION COMPLETE")
print("=" * 70)
print(f"  * Global Spatial R² Score       : {global_r2_val:.6f}")
print(f"  * Global Spatial MAE            : {global_mae_val:.4f} nm")
print(f"  * Global Spatial RMSE           : {global_rmse_val:.4f} nm")
print(f"  * Global Spatial Max Error      : {global_max_err:.4f} nm")
print(f"  * Global Mean MAPE              : {global_mape_val:.2f} %")
print(f"  * Conformality Prediction R²    : {conformality_r2:.6f}")
print(f"  * Conformality Prediction MAE   : {conformality_mae:.6f}")
print("=" * 70)


In [ ]:
import os
import glob
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import physicsnemo
from physicsnemo.models.mlp.fully_connected import FullyConnected

print("=" * 80)
print("INITIALIZING NATIVE NVIDIA PHYSX NEMO SPATIAL EVALUATION & DIAGNOSTICS")
print("=" * 80)

BASE_PATH = os.getcwd()

DATASET_DIR = os.path.join(BASE_PATH, "dataset")
MODEL_DIR = os.path.join(BASE_PATH, "models")
FIG_DIR = os.path.join(BASE_PATH, "figures")

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Working Directory: {BASE_PATH}")
print(f"Inference Device : {device}")

all_csvs = glob.glob(os.path.join(DATASET_DIR, "*.csv")) + glob.glob(os.path.join(BASE_PATH, "*.csv")) + glob.glob(os.path.join(BASE_PATH, "**", "*.csv"), recursive=True)

all_csvs = list(dict.fromkeys(all_csvs))

ignore_keywords = ["summary", "matrix", "history", "metrics", "eval", "outliers"]
candidate_csvs = [
    f for f in all_csvs
    if not any(kw in os.path.basename(f).lower() for kw in ignore_keywords)
]

DATASET_CSV = None
for csv_path in sorted(candidate_csvs, key=os.path.getmtime, reverse=True):
    try:
        temp_df = pd.read_csv(csv_path, nrows=5)
        if any(col.startswith("thickness_pt_") for col in temp_df.columns):
            DATASET_CSV = csv_path
            break
    except Exception:
        continue

if DATASET_CSV is None:
    raise FileNotFoundError(
        f"Could not locate raw spatial dataset CSV containing 'thickness_pt_*' columns in '{BASE_PATH}'."
    )

print(f"Loaded Dataset   : {DATASET_CSV}")

df = pd.read_csv(DATASET_CSV)
N_TOTAL_SAMPLES = len(df)

FEATURE_COLS = ["temperature_C", "pulse_time_s", "num_cycles", "tma_mole_fraction"]
TARGET_COLS = [col for col in df.columns if col.startswith("thickness_pt_")]
N_SPATIAL_POINTS = len(TARGET_COLS)

print(f"Detected {N_SPATIAL_POINTS} spatial target coordinates per sample across {N_TOTAL_SAMPLES} samples.")

bundle_path = os.path.join(MODEL_DIR, "ald_pinn_deployment_bundle.pt")
ckpt_path = os.path.join(MODEL_DIR, "ald_spatial_pinn_checkpoint.pt")

if os.path.exists(bundle_path):
    print(f"Restoring Model Bundle: {bundle_path}")
    checkpoint = torch.load(bundle_path, map_location=device, weights_only=False)
elif os.path.exists(ckpt_path):
    print(f"Restoring Checkpoint  : {ckpt_path}")
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
else:
    raise FileNotFoundError(f"No trained model checkpoint or deployment bundle found in {MODEL_DIR}")

if "preprocessing" in checkpoint:
    X_mean = checkpoint["preprocessing"]["X_mean"]
    X_std = checkpoint["preprocessing"]["X_std"]
    Y_mean = checkpoint["preprocessing"]["Y_mean"]
    Y_std = checkpoint["preprocessing"]["Y_std"]
else:
    X_mean = checkpoint["X_mean"]
    X_std = checkpoint["X_std"]
    Y_mean = checkpoint["Y_mean"]
    Y_std = checkpoint["Y_std"]

surrogate = FullyConnected(
    in_features=len(FEATURE_COLS),
    out_features=N_SPATIAL_POINTS,
    num_layers=6,
    layer_size=128,
).to(device)

surrogate.load_state_dict(checkpoint["model_state_dict"])
surrogate.eval()
print("Successfully loaded state weights into native PhysicsNeMo network architecture.")

X_raw = df[FEATURE_COLS].values.astype(np.float32)
Y_true_nm = df[TARGET_COLS].values.astype(np.float32)

X_norm = (X_raw - X_mean) / X_std
X_tensor = torch.tensor(X_norm, device=device)

with torch.no_grad():
    Y_pred_norm = surrogate(X_tensor).cpu().numpy()

Y_pred_nm = Y_pred_norm * Y_std + Y_mean

abs_error_matrix = np.abs(Y_true_nm - Y_pred_nm)
sample_mae_nm = abs_error_matrix.mean(axis=1)
sample_max_err_nm = abs_error_matrix.max(axis=1)

global_r2 = float(r2_score(Y_true_nm.flatten(), Y_pred_nm.flatten()))
global_mae = float(abs_error_matrix.mean())
global_rmse = float(np.sqrt(mean_squared_error(Y_true_nm, Y_pred_nm)))
global_max_err = float(abs_error_matrix.max())

pct_90_val = float(np.percentile(abs_error_matrix, 90))
pct_95_val = float(np.percentile(abs_error_matrix, 95))
pct_99_val = float(np.percentile(abs_error_matrix, 99))

tail_threshold_mae = np.percentile(sample_mae_nm, 95)
tail_indices = np.where(sample_mae_nm >= tail_threshold_mae)[0]

df_tail = df.iloc[tail_indices].copy()
df_tail["sample_mae_nm"] = sample_mae_nm[tail_indices]
df_tail["sample_max_err_nm"] = sample_max_err_nm[tail_indices]
df_tail["peak_error_point_index"] = abs_error_matrix[tail_indices].argmax(axis=1)

def categorize_point(pt_idx):
    if pt_idx < 6 or pt_idx > 93: return "Fin Peak (Unshadowed)"
    elif 40 <= pt_idx <= 60: return "Trench Bottom (Maximum Shadowing)"
    else: return "Fin Sidewall (Knudsen Transport Region)"

df_tail["peak_error_region"] = df_tail["peak_error_point_index"].apply(categorize_point)

correlations = {}
for col in FEATURE_COLS + ["gpc_nm_per_cycle", "conformality_bottom_to_top"]:
    if col in df.columns:
        r, _ = stats.pearsonr(df[col].values, sample_mae_nm)
        correlations[col] = float(r)

tail_csv_path = os.path.join(DATASET_DIR, "error_tail_outliers_top5pct.csv")
df_tail.to_csv(tail_csv_path, index=False)
print(f"[SAVED] Error tail outlier table written to: {tail_csv_path}")

fig, axes = plt.subplots(1, 3, figsize=(20, 6), gridspec_kw={'wspace': 0.15})
vmin = min(Y_true_nm.min(), Y_pred_nm.min())
vmax = max(Y_true_nm.max(), Y_pred_nm.max())

im0 = axes[0].imshow(Y_true_nm, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
axes[0].set_title("ViennaPS Ground Truth Thickness (nm)", fontweight='bold', fontsize=12)
axes[0].set_xlabel("Spatial Coordinate Index (0 = Left Fin, 50 = Bottom, 99 = Right Fin)")
axes[0].set_ylabel("Process Sample Index")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(Y_pred_nm, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
axes[1].set_title("NVIDIA PhysicsNeMo Predicted Thickness (nm)", fontweight='bold', fontsize=12)
axes[1].set_xlabel("Spatial Coordinate Index")
axes[1].set_yticks([])
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(abs_error_matrix, aspect='auto', cmap='inferno')
axes[2].set_title("Absolute Spatial Error |Y_true - Y_pred| (nm)", fontweight='bold', fontsize=12)
axes[2].set_xlabel("Spatial Coordinate Index")
axes[2].set_yticks([])
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.suptitle(f"ALD FinFET Spatial Topography Verification (N={N_TOTAL_SAMPLES}, Global R² = {global_r2:.6f})", fontsize=15, fontweight='bold')
heatmap_fig_path = os.path.join(FIG_DIR, f"spatial_heatmaps_full_n{N_TOTAL_SAMPLES}.png")
plt.savefig(heatmap_fig_path, dpi=200, bbox_inches='tight')
plt.close()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

flat_errors = abs_error_matrix.flatten()
ax1.hist(flat_errors, bins=100, color="#1f77b4", alpha=0.7, density=True, edgecolor="none")
ax1.axvline(pct_95_val, color="red", linestyle="--", linewidth=1.5, label=f"95th Percentile ({pct_95_val:.6f} nm)")
ax1.axvline(pct_99_val, color="darkred", linestyle=":", linewidth=2.0, label=f"99th Percentile ({pct_99_val:.6f} nm)")
ax1.set_yscale("log")
ax1.set_xlabel("Absolute Pointwise Spatial Error (nm)", fontweight='bold')
ax1.set_ylabel("Probability Density (Log Scale)", fontweight='bold')
ax1.set_title("Spatial Error Probability Density & Tail", fontweight='bold')
ax1.grid(True, which="both", alpha=0.3)
ax1.legend()

sorted_errs = np.sort(flat_errors)
cdf = np.arange(1, len(sorted_errs) + 1) / len(sorted_errs)
ax2.plot(sorted_errs, cdf, color="#2ca02c", linewidth=2.0)
ax2.axhline(0.95, color="red", linestyle="--", alpha=0.7, label="95% Coverage")
ax2.axvline(pct_95_val, color="red", linestyle="--", alpha=0.7)
ax2.set_xlabel("Absolute Spatial Error (nm)", fontweight='bold')
ax2.set_ylabel("Cumulative Probability", fontweight='bold')
ax2.set_title("Cumulative Error Distribution (CDF)", fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
tail_fig_path = os.path.join(FIG_DIR, f"error_tail_distribution_n{N_TOTAL_SAMPLES}.png")
plt.savefig(tail_fig_path, dpi=200, bbox_inches='tight')
plt.close()

fig, ax = plt.subplots(figsize=(10, 5))
spatial_x = np.arange(N_SPATIAL_POINTS)
pt_mae = abs_error_matrix.mean(axis=0)
pt_std = abs_error_matrix.std(axis=0)
pt_max = abs_error_matrix.max(axis=0)

ax.plot(spatial_x, pt_mae, color="#d62728", linewidth=2.0, label="Pointwise MAE")
ax.fill_between(spatial_x, np.maximum(0, pt_mae - pt_std), pt_mae + pt_std, color="#d62728", alpha=0.2, label="±1 Std Dev")
ax.plot(spatial_x, pt_max, color="black", linestyle=":", linewidth=1.5, label="Max Envelope Error")

ax.axvspan(0, 5, color="gray", alpha=0.15, label="Fin Peak")
ax.axvspan(94, 99, color="gray", alpha=0.15)
ax.axvspan(40, 60, color="blue", alpha=0.1, label="Trench Bottom")

ax.set_xlabel("Spatial Coordinate Index Across Trench Domain", fontweight='bold')
ax.set_ylabel("Absolute Error (nm)", fontweight='bold')
ax.set_title("Pointwise Spatial Error Breakdown Along FinFET Trench Geometry", fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right")

profile_fig_path = os.path.join(FIG_DIR, f"pointwise_spatial_error_profile_n{N_TOTAL_SAMPLES}.png")
plt.savefig(profile_fig_path, dpi=200, bbox_inches='tight')
plt.close()

print(f"[SAVED] Spatial heatmaps saved to  : {heatmap_fig_path}")
print(f"[SAVED] Error tail figure saved to  : {tail_fig_path}")
print(f"[SAVED] Spatial profile saved to    : {profile_fig_path}")

print("\n" + "=" * 80)
